# Managing properties

In this notebook, we cover how to work with the properties of a Pycellin model:
- the different kinds of properties,
- how to add predefined Pycellin properties to a model,
- how to compute their values by updating the model,
- how to choose the input of a property computed from another one,
- how to propagate cell cycle properties to the cells,
- how to remove properties from a model.

It is recommended to first read the [Getting started](./Getting%20started.ipynb) notebook. Defining your own properties is covered in the [Custom properties](./Custom%20properties.ipynb) notebook.

In [1]:
import pycellin as pc

The data used throughout the notebook comes from the tracking of fluorescently labeled *Escherichia coli* bacteria growing on agar pad. The tracking data was obtained with TrackMate and then exported as an XML file.

In [2]:
# Path to the TrackMate XML file.
xml_path = "../sample_data/Ecoli_growth_on_agar_pad.xml"

# Parse the XML file and create a Pycellin Model object
# that contains all the data from the XML file.
model = pc.read_trackmate_xml(xml_path)

# We can display basic information about this model.
print(model)
print(f"This model contains {model.data.number_of_lineages()} lineages:")
for lin_id, lineage in model.data.cell_lineages.items():
    print(f"- ID {lin_id}: {lineage}")

Model named 'Ecoli_growth_on_agar_pad' with 3 lineages, built from TrackMate.
This model contains 3 lineages:
- ID 0: CellLineage of ID 0 with 152 cells and 151 links.
- ID 1: CellLineage of ID 1 with 189 cells and 188 links.
- ID 2: CellLineage of ID 2 with 185 cells and 184 links.


## What is a property in Pycellin?

A property is any information that can be stored into a lineage and that is related to part or all of the lineage, e.g. the shape of a cell, the speed of a cell or the number of divisions in a lineage. Its values are stored on the cells (nodes), the links (edges) or the lineages themselves (graphs), as shown in the "Pycellin model structure" section of the [Getting started](./Getting%20started.ipynb) notebook.

Property values can be imported with the data, computed by Pycellin, or added manually. Depending on where they come from, there are 3 kinds of properties.

### Imported properties

Imported properties come with data created by an external tool, e.g. TrackMate. They are added automatically when the data is loaded, but Pycellin cannot recompute them.

### Pycellin properties

Pycellin properties are predefined properties that you can add to a model as needed, and that Pycellin computes:
- morphological properties, e.g. the length of rod-shaped bacteria
- intensity properties, e.g. the mean intensity of a cell
- tracking related properties, e.g. the interdivision time
- motion related properties, e.g. the speed of a cell

### Custom properties

Custom properties are defined by you, usually for needs more specific to an experiment or a type of data. They are more advanced and require more Python knowledge: see the [Custom properties](./Custom%20properties.ipynb) notebook.

## Adding properties to a model

When you add a property to a model, it lets the model know which properties are expected on the lineages.
It is also a way for the user to have a clear description of the properties (provenance, units, the type of data it applies to...). However, properties do not have values at this stage: the properties are just *declared*, not *computed*. In the model, the information about the added properties are stored in the PropsMetadata of the model:


In [3]:
print("Identifiers of all the properties already declared in the model:")
print(model.props_metadata)
print()

print("Detailed information on a few of them:")
for prop_id in ["AREA", "SPEED", "TRACK_DURATION"]:
    print(model.get_property(prop_id))

Identifiers of all the properties already declared in the model:
Node properties: QUALITY, POSITION_T, FRAME, RADIUS, VISIBILITY, MANUAL_SPOT_COLOR, MEAN_INTENSITY_CH1, MEDIAN_INTENSITY_CH1, MIN_INTENSITY_CH1, MAX_INTENSITY_CH1, TOTAL_INTENSITY_CH1, STD_INTENSITY_CH1, MEAN_INTENSITY_CH2, MEDIAN_INTENSITY_CH2, MIN_INTENSITY_CH2, MAX_INTENSITY_CH2, TOTAL_INTENSITY_CH2, STD_INTENSITY_CH2, CONTRAST_CH1, SNR_CH1, CONTRAST_CH2, SNR_CH2, ELLIPSE_X0, ELLIPSE_Y0, ELLIPSE_MAJOR, ELLIPSE_MINOR, ELLIPSE_THETA, ELLIPSE_ASPECTRATIO, AREA, PERIMETER, CIRCULARITY, SOLIDITY, SHAPE_INDEX, cell_name, cell_id, cell_x, cell_y, cell_z, cell_contour, lineage_id, timepoint
Edge properties: SPOT_SOURCE_ID, SPOT_TARGET_ID, LINK_COST, DIRECTIONAL_CHANGE_RATE, SPEED, DISPLACEMENT, EDGE_TIME, MANUAL_EDGE_COLOR, link_x, link_y, link_z
Lineage properties: TRACK_INDEX, DIVISION_TIME_MEAN, DIVISION_TIME_STD, NUMBER_SPOTS, NUMBER_GAPS, NUMBER_SPLITS, NUMBER_MERGES, NUMBER_COMPLEX, LONGEST_GAP, TRACK_DURATION, TRACK_STA

Here you can see that there are already a lot of properties declared in the model. These properties were imported from TrackMate when the XML file was loaded in Pycellin (see code in the second cell, at the top of the notebook). Depending on the goal of your analysis, you may want to add properties more relevant to your study, like growth rate or division rate of your cells.
Of course, if you build a Pycellin model from scratch instead of importing a pre-existing tracking file (from TrackMate or another tracking tool), then your model will first be devoid of properties.

In this part, we will only see how to add pre-defined Pycellin properties to a model. Indeed, imported properties are automatically added during importation, and custom properties are more complex and are covered in the [Custom properties](./Custom%20properties.ipynb) notebook.

There are 2 types of lineages in Pycellin: cell lineages and cell cycle lineages (cycle lineages for short). Each lineage type has its own properties. If you are unsure about the difference between a cell lineage (`CellLineage`) and a cell cycle lineage (`CycleLineage`), I invite you to read the "Pycellin model structure" section of the [Getting started](./Getting%20started.ipynb) notebook before going further.


### For cell lineages

In [4]:
lin0 = model.data.cell_lineages[0]
lin0.plot(
    title="Cell lineage of lineage ID 0",
    node_colormap_prop="AREA",
    node_colormap="Plotly3",
    node_hover_props=["AREA", "CIRCULARITY"],
    node_marker_style={
        "size": 10,
    },
    width=800,
    height=600,
    show_legend=False,
)

List of available Pycellin properties for cell lineages, without the core properties, like `cell_id` or `timepoint`, which Pycellin sets by itself:

In [5]:
pc.get_pycellin_cell_lineage_properties(include_core_properties=False)

{'cell_total_intensity': 'Sum of the pixel values of the cell in an intensity image',
 'cell_mean_intensity': 'Mean of the pixel values of the cell in an intensity image',
 'cell_polygon': 'Cell shape as a shapely.Polygon',
 'cell_multipolygon': 'Cell shape as a shapely.MultiPolygon, with all the pieces and holes of the cell label, computed from a label image',
 'cell_area': 'Area of the cell',
 'cell_contour': "List of coordinates of the cell's contour, relative to its centroid",
 'cell_perimeter': 'Perimeter of the cell',
 'cell_circularity': 'Circularity of the cell, 4 pi times its area divided by its squared perimeter: 1 for a disk, closer to 0 for an elongated or irregular cell',
 'cell_solidity': 'Solidity of the cell, its area divided by the area of its convex hull: 1 for a convex cell, closer to 0 for a more concave cell',
 'rod_width': 'Width of the cell, for rod-shaped cells only',
 'rod_length': 'Length of the cell, for rod-shaped cells only',
 'cell_displacement': 'Displace

You can add properties one by one, either by calling their dedicated method:

In [6]:
model.add_absolute_age()

Or by using the more generic `add_pycellin_property()` method with the identifier of the property (adding a property that is already in the model replaces it, with a warning):

In [7]:
model.add_pycellin_property("relative_age")

When you want to add a lot of properties, it might be easier to add them all at once with the `add_pycellin_properties()` shortcut method that accepts a list of Pycellin property identifiers:

In [8]:
model.add_pycellin_properties(["rod_length", "normalized_age"])

At any time, you can check which cell lineage properties are currently present in the model, regardless of their provenance (imported, pre-defined by Pycellin or custom built):

In [9]:
list(model.get_cell_lineage_properties())

['QUALITY',
 'POSITION_T',
 'FRAME',
 'RADIUS',
 'VISIBILITY',
 'MANUAL_SPOT_COLOR',
 'MEAN_INTENSITY_CH1',
 'MEDIAN_INTENSITY_CH1',
 'MIN_INTENSITY_CH1',
 'MAX_INTENSITY_CH1',
 'TOTAL_INTENSITY_CH1',
 'STD_INTENSITY_CH1',
 'MEAN_INTENSITY_CH2',
 'MEDIAN_INTENSITY_CH2',
 'MIN_INTENSITY_CH2',
 'MAX_INTENSITY_CH2',
 'TOTAL_INTENSITY_CH2',
 'STD_INTENSITY_CH2',
 'CONTRAST_CH1',
 'SNR_CH1',
 'CONTRAST_CH2',
 'SNR_CH2',
 'ELLIPSE_X0',
 'ELLIPSE_Y0',
 'ELLIPSE_MAJOR',
 'ELLIPSE_MINOR',
 'ELLIPSE_THETA',
 'ELLIPSE_ASPECTRATIO',
 'AREA',
 'PERIMETER',
 'CIRCULARITY',
 'SOLIDITY',
 'SHAPE_INDEX',
 'cell_name',
 'SPOT_SOURCE_ID',
 'SPOT_TARGET_ID',
 'LINK_COST',
 'DIRECTIONAL_CHANGE_RATE',
 'SPEED',
 'DISPLACEMENT',
 'EDGE_TIME',
 'MANUAL_EDGE_COLOR',
 'TRACK_INDEX',
 'DIVISION_TIME_MEAN',
 'DIVISION_TIME_STD',
 'NUMBER_SPOTS',
 'NUMBER_GAPS',
 'NUMBER_SPLITS',
 'NUMBER_MERGES',
 'NUMBER_COMPLEX',
 'LONGEST_GAP',
 'TRACK_DURATION',
 'TRACK_START',
 'TRACK_STOP',
 'TRACK_DISPLACEMENT',
 'TRACK_ME

Most of Pycellin properties are expressed in time and/or space units. These units are defined at the model level, usually during the model creation. In most cases, the exact units of a specific Pycellin property will only be known once the property has been added to the model.  

Information on a property, units included, is accessible through the PropsMetadata of the model, regardless of the property provenance:

In [10]:
print(model.get_property("absolute_age"))  # Pycellin property.
print(model.get_property("SPEED"))  # Imported property.

Property 'absolute_age'
  Name: Absolute age
  Description: Age of the cell since the start of the lineage
  Provenance: pycellin
  Type: node
  Lineage type: cell
  Data type: float
  Unit: minute
Property 'SPEED'
  Name: Speed
  Description: Speed
  Provenance: TrackMate
  Type: edge
  Lineage type: cell
  Data type: float
  Unit: micrometer/minute


### For cell cycle lineages

Cell cycle properties (or cycle properties for short) behave like cell properties. However, you first need to compute the cycle lineages, otherwise you will get the following error:

In [11]:
model.add_interdivision_time()

ValueError: Cycle lineages have not been computed yet. Please compute them first with model.add_cycle_lineages().

To compute and add the cycle lineages to the model, just call the `add_cycle_lineages()` method on the model:

In [12]:
model.add_cycle_lineages()

A few mandatory cycle lineage properties are automatically computed:
- lineage_id: ID of the lineage, the same as that of the cell lineage;
- cycle_id: ID of the cell cycle, i.e. cell ID of the last cell in the cell cycle;
- cells: cell IDs of the cells in the cell cycle, in chronological order;
- first_cell: cell ID of the first cell in the cell cycle;
- last_cell: cell ID of the last cell in the cell cycle, i.e. its cycle_id;
- cycle_num_cells: number of cells in the cell cycle, not counting gaps (missing detections);
- cycle_duration: duration of the cell cycle, regardless of gaps;
- cycle_start_time: time of the first cell in the cell cycle;
- cycle_end_time: time of the last cell in the cell cycle;
- generation: generation of the cell cycle in the lineage, i.e. number of cell cycles upstream of the current one.

In [13]:
cyclelin0 = model.data.cycle_lineages[0]
cyclelin0.plot(
    title="Cycle lineage of lineage ID 0",
    node_hover_props=["cycle_id", "generation", "cycle_num_cells"],
    node_marker_style={"size": 10},
    width=800,
    height=300,
    show_legend=False,
)

List of available Pycellin properties for cell cycle lineages, without the core properties listed above:

In [14]:
pc.get_pycellin_cycle_lineage_properties(include_core_properties=False)

{'cycle_mean_intensity': 'Mean intensity of the cell during the cell cycle',
 'birth_intensity': 'Intensity of the cell at the start of the cell cycle, right after division',
 'division_intensity': 'Intensity of the cell at the end of the cell cycle, right before division',
 'cycle_mean_area': 'Mean area of the cell during the cell cycle',
 'birth_area': 'Area of the cell at the start of the cell cycle, right after division',
 'division_area': 'Area of the cell at the end of the cell cycle, right before division',
 'cycle_total_displacement': 'Displacement of the cell during the cell cycle',
 'cycle_mean_displacement': 'Mean displacement of the cell during the cell cycle',
 'cycle_mean_speed': 'Mean speed of the cell during the cell cycle',
 'straightness': 'Straightness of the cell trajectory during the cell cycle',
 'interdivision_time': 'Time elapsed between two successive divisions',
 'division_rate': 'Number of divisions per time unit',
 'is_complete_cycle': 'Whether the full cell

Like for cell lineages, you can add new properties either one by one by calling the method specific to your property:

In [15]:
model.add_interdivision_time()

Or the generic `add_pycellin_property()`:

In [16]:
model.add_pycellin_property("is_complete_cycle")

Or several properties at once with `add_pycellin_properties()`:

In [17]:
model.add_pycellin_properties(["division_rate", "num_cycles"])

Cell lineage and cycle lineage properties can be added at the same time without issue, given that cycle lineage data has already been computed:

In [18]:
model.add_pycellin_properties(
    [
        "cell_displacement",  # cell lineage property
        "cycle_total_displacement",  # cycle lineage property
        "cell_speed",  # cell lineage property
    ]
)

To check which cycle lineage properties are currently present in the model:

In [19]:
list(model.get_cycle_lineage_properties())

['cycle_id',
 'cells',
 'first_cell',
 'last_cell',
 'cycle_num_cells',
 'cycle_duration',
 'cycle_start_time',
 'cycle_end_time',
 'generation',
 'interdivision_time',
 'is_complete_cycle',
 'division_rate',
 'num_cycles',
 'cycle_total_displacement',
 'lineage_id']

### For optional arguments

Some properties accept optional arguments, often to tweak the way the property is defined and computed: change of unit, change of algorithm...

When adding properties one by one, additional arguments are given like classic optional argument.

For example, with property specific methods:

``` python
model.add_rod_length(skeleton_method="lee", tolerance=0.6)
model.add_turning_angle(unit="degree")
```

And with the generic `add_pycellin_property()`:

``` python
model.add_pycellin_property("rod_length", skeleton_method="lee", tolerance=0.6)
model.add_pycellin_property("turning_angle", unit="degree")
```

When adding several properties at the same time, each property with additional arguments is given in the list as a dictionary, with the name of the property as the key and its keyword arguments as the value:

``` python
model.add_pycellin_properties(
    [
        {"rod_length": {"skeleton_method": "lee", "tolerance": 0.6}},
        {"turning_angle": {"unit": "degree"}},
    ]
)
```

It is also possible to mix properties with and without additional arguments:

``` python
model.add_pycellin_properties(
    [{"turning_angle": {"unit": "degree"}}, "is_complete_cycle"]
)
```

## Computing or recomputing properties

Now that the Pycellin properties we are interested in have been declared, we want to actually compute their values to enrich the lineages. To do this, we need to ask the model to perform an update:

In [20]:
model.update()

More generally, an update computes or recomputes all the needed properties on the impacted structural elements of the model (cells, cycles (if cycle lineages are present), links and lineages).

We can now check that the values have been computed and stored in the lineage by directly accessing the property value of a specific cell or cell cycle:

In [21]:
# For the cell lineage
print(lin0.nodes[9013]["relative_age"])
print(lin0.nodes[9013]["rod_length"])

# For the cycle lineage
# After an update, references to the cycle lineage are lost,
# so we cannot use cyclelin0 anymore.
print(model.get_cycle_lineage(0).nodes[9019]["interdivision_time"])

0.0
2.7544300640764128
30.0


Or by plotting a lineage of the model and asking for the property of interest when hovering over the lineage elements:

In [22]:
lin0.plot(
    title="Cell lineage of lineage ID 0 with new properties",
    node_hover_props=["cell_id", "relative_age", "rod_length"],
    node_marker_style={"size": 10},
    width=800,
    height=600,
    show_legend=False,
)

In [23]:
model.get_cycle_lineage(0).plot(
    title="Cycle lineage of lineage ID 0 with new properties",
    node_hover_props=["cycle_id", "interdivision_time", "is_complete_cycle"],
    node_marker_style={"size": 10},
    width=800,
    height=300,
    show_legend=False,
)

### When is an update required?

An update is required in 2 different cases:
- When you have added one or more properties to a model. In that case, the added properties will have no associated values and the update is needed to compute them. All cells, cycles (if cycle lineages are present), links and lineages will be affected by the update.
- When the structure of a lineage has been modified, e.g. removal of a cell or correction of an inaccurate division. In that case, the properties already have values but they may be incorrect. Depending on the reach of the properties and the impacted elements of the model, only a subset of cells, cycles (if cycle lineages are present), links and lineages will be affected by the update.

You can check if an update is required with:

In [24]:
model.is_update_required()

False

However, there is one case not taken into account by `update()` and `is_update_required()`: **when the value of a property is manually modified on one or more elements**, which in turn should impact the value of another property.
A classic example is when the shape of a specific cell is modified by the user. Since cell length depends on the shape of the cell, length for this specific cell becomes incorrect. In that case, call `prepare_full_data_update()` before `update()`: the update then recomputes the properties on all the cells, links and lineages.

## Properties computed from other properties

Some Pycellin properties are computed from another property: for example, the cycle mean speed is the mean of the cell speeds of the cell cycle, and the cell area is computed from the cell polygon. For these properties:
- by default, Pycellin uses its own property (here `cell_speed` and `cell_polygon`);
- you can use any other declared property instead by passing its identifier, e.g. `speed_prop="SPEED"` to use the speeds computed by TrackMate;
- if the input property has not been declared, adding the property raises an error that says how to fix it. Missing input properties are never added automatically.

| Properties | Parameter | Default input | Added by |
|---|---|---|---|
| `cell_area`, `cell_perimeter`, `cell_circularity`, `cell_solidity`, `cell_contour`; `cell_total_intensity`, `cell_mean_intensity` with `source="polygon"` | `polygon_prop` | `cell_polygon` | `add_cell_polygon()` |
| `cycle_mean_area`, `birth_area`, `division_area` | `area_prop` | `cell_area` | `add_cell_area()` |
| `cycle_mean_displacement`, `cycle_total_displacement` | `displacement_prop` | `cell_displacement` | `add_cell_displacement()` |
| `cycle_mean_speed` | `speed_prop` | `cell_speed` | `add_cell_speed()` |
| `cycle_mean_intensity` | `intensity_prop` | `cell_mean_intensity` | `add_cell_mean_intensity()` |
| `birth_intensity`, `division_intensity` | `intensity_prop` | `cell_total_intensity` | `add_cell_total_intensity()` |
| `cell_polygon`, `cell_total_intensity`, `cell_mean_intensity` with `source="label_image"`; `cell_multipolygon` | `label_prop` | `label` | your data, or you: no Pycellin reader creates it |
| `cell_polygon` with `source="contour"` | `contour_prop` | `cell_contour` | your loader (TrackMate, CTC) |
| `absolute_age`, `relative_age`, `normalized_age`, `cell_speed`, `interdivision_time`, `division_rate`, `lineage_duration` | `time_prop` | the reference time property of the model | your loader |

The unit of the new property comes from its input property.

### Using Pycellin's own input property

We added `cell_speed` earlier, so the cycle mean speed can use it without any argument:

In [25]:
model.add_cycle_mean_speed()
model.update()
model.to_cycle_dataframe()[["cycle_id", "cycle_num_cells", "cycle_mean_speed"]].head()

,cycle_id,cycle_num_cells,cycle_mean_speed
0,8993,1,NaN
1,9019,6,0.054189
2,9020,6,0.102414
3,9051,5,0.046958
4,9052,5,0.051218


When the input property is missing, the error says what to do. For example, the cycle mean area needs the cell area, which is not in the model yet:

In [26]:
model.add_cycle_mean_area()

MissingPropertyError: 'cycle_mean_area' requires the property 'cell_area', which has not been declared. To fix this, add it first with model.add_cell_area(), or set 'area_prop' to the identifier of a declared property.

The cell area is itself computed from the cell polygon. By default, Pycellin computes cell polygons from a label image, and we don't have one for this dataset. But TrackMate gives the contour of each cell (`cell_contour`, relative to the cell position), so we can build the polygons from it with `source="contour"`.

Since each property is added after its input property, a single update computes them all in the right order. The cell area computed by Pycellin matches the `AREA` computed by TrackMate:

In [27]:
model.add_cell_polygon(source="contour")
model.add_cell_area()
model.add_cycle_mean_area()
model.update()
model.to_cell_dataframe()[["cell_id", "AREA", "cell_area"]].head()

,cell_id,AREA,cell_area
0,8993,2.666228,2.666228
1,9013,1.282132,1.282132
2,9014,1.262608,1.262608
3,8986,1.462195,1.462195
4,8989,1.227897,1.227897


### Using a property from the loader

TrackMate computed its own speeds (`SPEED`, a link property) and areas (`AREA`, a cell property). Pass their identifiers to use them instead of Pycellin's. Here we also give the new properties a custom identifier, to keep them next to the ones computed from Pycellin's inputs. The generic `add_pycellin_property()` and `add_pycellin_properties()` methods accept the same arguments:

In [28]:
model.add_cycle_mean_area(area_prop="AREA", custom_identifier="cycle_mean_area_TM")
model.add_pycellin_property(
    "cycle_mean_speed", speed_prop="SPEED", custom_identifier="cycle_mean_speed_TM"
)
model.update()
model.to_cycle_dataframe()[
    [
        "cycle_id",
        "cycle_mean_area",
        "cycle_mean_area_TM",
        "cycle_mean_speed",
        "cycle_mean_speed_TM",
    ]
].head()

,cycle_id,cycle_mean_area,cycle_mean_area_TM,cycle_mean_speed,cycle_mean_speed_TM
0,8993,2.666228,2.666228,NaN,NaN
1,9019,1.768084,1.768084,0.054189,0.054189
2,9020,2.063488,2.063488,0.102414,0.102414
3,9051,2.117362,2.117362,0.046958,0.046958
4,9052,2.573810,2.573810,0.051218,0.051218


The unit of each new property comes from its input property:

In [29]:
for prop_id in [
    "cell_speed",
    "cycle_mean_speed",
    "SPEED",
    "cycle_mean_speed_TM",
    "cell_area",
    "cycle_mean_area",
]:
    print(f"{prop_id:20} {model.get_property(prop_id).unit}")

cell_speed           micrometer/minute
cycle_mean_speed     micrometer/minute
SPEED                micrometer/minute
cycle_mean_speed_TM  micrometer/minute
cell_area            micrometer^2
cycle_mean_area      micrometer^2


### Using a shape you computed

For a shape Pycellin doesn't compute, write a calculator that returns a shapely polygon (see the [Custom properties](./Custom%20properties.ipynb) notebook), then pass its identifier to the shape properties. Here, the convex hull of each cell gives its convex area. The ratio of the area to the convex area, the solidity, is a Pycellin property: `model.add_cell_solidity()`.

In [30]:
class ConvexHull(pc.NodeLocalPropCalculator):
    def compute(self, lineage, node_id):
        return lineage.nodes[node_id]["cell_polygon"].convex_hull


convex_hull_prop = pc.Property(
    identifier="convex_hull",
    name="Convex hull",
    description="Convex hull of the cell polygon",
    provenance="custom",
    prop_type="node",
    lineage_type="cell",
    dtype="shapely.Polygon",
    unit=model.get_space_unit(),
)
model.add_custom_property(ConvexHull(convex_hull_prop))
model.add_cell_area(polygon_prop="convex_hull", custom_identifier="convex_area")
model.update()
model.to_cell_dataframe()[["cell_id", "cell_area", "convex_area"]].head()

,cell_id,cell_area,convex_area
0,8993,2.666228,3.121808
1,9013,1.282132,1.399282
2,9014,1.262608,1.431823
3,8986,1.462195,1.596700
4,8989,1.227897,1.457856


### Mistakes caught when adding a property

The input property is checked when you add the property, not later when you call `update()`.

A typo in the identifier:

In [31]:
model.add_division_area(area_prop="AERA")

MissingPropertyError: 'division_area' requires the property 'AERA', which has not been declared. To fix this, set 'area_prop' to the identifier of a declared property.

A cell property where a link property is expected:

In [32]:
model.add_cycle_mean_speed(speed_prop="AREA")

ValueError: 'cycle_mean_speed' requires 'speed_prop' to be a property with prop_type 'edge', but 'AREA' has prop_type 'node'.

A cycle lineage property where a cell lineage property is expected:

In [33]:
model.add_birth_area(area_prop="cycle_duration")

ValueError: 'birth_area' requires 'area_prop' to be a property with lineage_type 'cell', but 'cycle_duration' has lineage_type 'cycle'.

An argument that doesn't apply to the chosen source of the cell polygon:

In [34]:
model.add_cell_polygon(source="contour", label_prop="label")

ValueError: 'label_prop' is not used with source='contour'.

To declare the input properties of your own properties, see the [Custom properties](./Custom%20properties.ipynb) notebook.

## Propagation of cycle lineage properties

In most cases, properties are either defined on cell lineages (age of a cell, area...) or on cycle lineages (completeness of the cell cycle, interdivision time...). But in some cases, you may want to compute a property on a cycle lineage but have the data stored on both the cycle lineage and the cell lineage, i.e. on the cell cycle itself but also on each cell of the cell cycle.

For example, you may want to obtain a dataframe of cell properties for plotting purposes, but you want to ignore incomplete cell cycles. With `propagate_cycle_properties()`, you can propagate the `is_complete_cycle` property, which is normally a cycle lineage property, to the cell lineages, and then filter the complete cell cycles.

In [35]:
print("Propagated properties before:", list(model.get_propagated_properties()))
# The `props` argument allows to specify which properties to propagate.
# When not specified, all the cycle properties are propagated.
model.propagate_cycle_properties(props=["is_complete_cycle"])
print("Propagated properties after:", list(model.get_propagated_properties()))
# A propagated property stays a cycle lineage property.
print("Lineage type:", model.get_property("is_complete_cycle").lineage_type)
print()

df = model.to_cell_dataframe()
print("Total number of cells:", df.shape[0])
df_complete_cc = df[df["is_complete_cycle"]]
print("Number of cells with complete cycle:", df_complete_cc.shape[0])

Propagated properties before: []
Propagated properties after: ['is_complete_cycle']
Lineage type: cycle

Total number of cells: 526
Number of cells with complete cycle: 322


In the 2 plots below, nodes are represented in blue when they belong to a complete cell cycle, and in red otherwise.

In [36]:
clin0 = model.get_cycle_lineage(0)
clin0.plot(
    title="Cycle lineage of lineage ID 0 with cycle completeness",
    node_hover_props=["cycle_id", "is_complete_cycle", "cells"],
    node_colormap_prop="is_complete_cycle",
    node_colormap="Bluered_r",
    node_marker_style={"size": 10},
    width=800,
    height=300,
    show_legend=False,
)

In [37]:
lin0.plot(
    title=(
        "Cell lineage of lineage ID 0 with cycle completeness propagated from the "
        "cycle lineage"
    ),
    node_hover_props=["cell_id", "is_complete_cycle"],
    node_colormap_prop="is_complete_cycle",
    node_colormap="Bluered_r",
    node_marker_style={"size": 10},
    width=800,
    height=600,
    show_legend=False,
)

When the model is updated, the propagated properties are computed again on the cycle lineages and copied again to the cells, so they stay up to date, e.g. after removing a link.

**WARNING:** Be careful when analyzing data with propagated properties, you may introduce a bias. Since the value of a property gets propagated on each cell of a cell cycle, this value will be overrepresented for long cell cycles compared to short ones. For example, depending on how you pool your data, boxplots are at risk of being biased.

## Removing properties from a model

Removing a property both removes all the property values stored in the lineages and unregisters the property itself from the model. It is a non-reversible operation. If you want to get a Pycellin or custom property back, the only way is to re-add it to the model, then update the model to recompute all the values. An imported property can only be recovered by loading the data again.

The properties Pycellin needs to work, like `cell_id`, `lineage_id`, `timepoint` or the reference time property of the model, are protected: removing them raises a `ProtectedPropertyError`.

For now, properties must be removed one by one:

In [38]:
model.remove_property("absolute_age")
model.remove_property("DISPLACEMENT")
model.remove_property("LONGEST_GAP")

After removal, trying to access the property values will throw an error:

In [39]:
lin0.nodes[9013]["absolute_age"]

KeyError: 'absolute_age'